# Root Cause Investigation Workflow

This notebook demonstrates systematic root cause analysis following a 50% revenue drop. Rather than relying on guesswork or surface-level assumptions, we use evidence-based data filtering: **Narrow Time → Narrow Segment → Find Pattern → Formulate Hypothesis → Validate**.

## Setup & Data Ingestion

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
from scripts.root_cause_investigation import (
    load_data,
    isolate_time_window,
    analyze_segments,
    analyze_correlations,
    generate_investigation_report,
    visualize_investigation
)

RAW_DATA_PATH = "../data/raw/investigation_telemetry.csv"
df = load_data(RAW_DATA_PATH)
print(f"Loaded {len(df):,} telemetry logs.")
df.head()

## Task 1: Isolate Time Window

We calculate daily success rates across the dataset to pinpoint anomaly dates below normal operational thresholds, then zoom in to hourly breakdowns to isolate the exact problem window.

In [ ]:
problem_day, problem_hour, hourly_df = isolate_time_window(df)

print(f"\n=== ANOMALY TIME ISOLATION ===")
print(f"Anomaly Date Detected: {problem_day}")
print(f"Worst Hour Window:     {problem_hour}:00 - {problem_hour+1}:00 UTC")
print(f"Success Rate Drop:     {float(hourly_df.loc[problem_hour, 'success_rate']):.1%}")

## Task 2: Segment Analysis

Next, we isolate the problem window (`2026-07-18` at hour `14:00 UTC`) and break down failure rates across multiple system dimensions: Customer Type, Payment Method, Region, and Device Type.

In [ ]:
affected_segment, by_payment = analyze_segments(df, problem_day, problem_hour)

print("\n=== SEGMENT BREAKDOWN BY PAYMENT METHOD ===")
print(by_payment)
print(f"\nPrimary Failure Concentration: payment_method = '{affected_segment}'")

## Task 3: Correlation Analysis & Error Profiling

We evaluate contingency tables and profile error messages during the problem window to verify if failures correlate with external gateway issues.

In [ ]:
top_error, error_pct = analyze_correlations(df, problem_day, problem_hour)

print(f"\n=== DOMINANT ERROR CODE ===")
print(f"Top Error Message: '{top_error}'")
print(f"Failure Concentration: {error_pct:.1%}")

## Task 4 & 5: Investigation Documentation & Hypothesis Validation

We formulate a formal Root Cause Investigation Report detailing the observation, evidence, hypothesis (Confidence: HIGH), external timeline validation, and strategic failover recommendations.

In [ ]:
report = generate_investigation_report(
    df,
    problem_day,
    problem_hour,
    affected_segment,
    top_error,
    error_pct,
    output_path="../output/investigation_report.txt"
)
print(report)

## Timeline Visualization

We render the hourly transaction success rate timeline on the problem day, contrasting credit card traffic against other payment methods.

In [ ]:
visualize_investigation(df, problem_day, problem_hour, output_path="../output/investigation_timeline.png")